# Lab 4: Conversational Search

**Duration:** 35 minutes  
**Environment:** Azure ML Studio Notebook  

---

## Learning Objectives

- Implement conversational search with Azure AI Search
- Compare search modes: keyword, vector, hybrid
- Build a multi-turn search experience with filters
- Use query rewriting for follow-up questions

## Prerequisites

Labs 1 + 2 + 3 completed

---

© 2026 Great Learning. All rights reserved.

## Step 1: Load Configuration

In [ ]:
import os, json, urllib.request, ssl, subprocess

RESOURCE_GROUP = 'rg-promptflow-rag-lab'

# Auto-detect OpenAI resource that has BOTH gpt-4o and text-embedding-3-small deployments
# This handles the case where Lab 1 was run multiple times creating multiple OpenAI resources
print('Detecting Azure resources...')
r = subprocess.run(['az', 'cognitiveservices', 'account', 'list', '-g', RESOURCE_GROUP,
    '--query', "[?kind=='OpenAI'].name", '-o', 'tsv'], capture_output=True, text=True)
openai_candidates = [n.strip() for n in r.stdout.strip().split('\n') if n.strip()]

OPENAI_NAME = ''
for candidate in openai_candidates:
    # Check if this resource has both required deployments
    r = subprocess.run(['az', 'cognitiveservices', 'account', 'deployment', 'list',
        '-n', candidate, '-g', RESOURCE_GROUP, '--query', '[].name', '-o', 'tsv'],
        capture_output=True, text=True)
    deployments = r.stdout.strip().split('\n')
    if 'gpt-4o' in deployments and 'text-embedding-3-small' in deployments:
        OPENAI_NAME = candidate
        print(f'  Found OpenAI with deployments: {candidate}')
        break

if not OPENAI_NAME:
    print('ERROR: No OpenAI resource found with both gpt-4o and text-embedding-3-small deployments.')
    print(f'  Candidates checked: {openai_candidates}')
    print('  Fix: Go back to Lab 1 and run Steps 4 + 5 to create deployments.')
    raise SystemExit(1)

# Auto-detect Search service
r = subprocess.run(['az', 'search', 'service', 'list', '-g', RESOURCE_GROUP,
    '--query', '[].name', '-o', 'tsv'], capture_output=True, text=True)
search_candidates = [n.strip() for n in r.stdout.strip().split('\n') if n.strip()]
SEARCH_NAME = search_candidates[0] if search_candidates else ''

# Auto-detect Storage account
r = subprocess.run(['az', 'storage', 'account', 'list', '-g', RESOURCE_GROUP,
    '--query', '[0].name', '-o', 'tsv'], capture_output=True, text=True)
STORAGE_NAME = r.stdout.strip()

if not SEARCH_NAME:
    print('ERROR: No Search service found. Run Lab 1 first.')
    raise SystemExit(1)

# Get endpoints and keys
r = subprocess.run(['az', 'cognitiveservices', 'account', 'show', '-n', OPENAI_NAME, '-g', RESOURCE_GROUP,
    '--query', 'properties.endpoint', '-o', 'tsv'], capture_output=True, text=True)
OPENAI_ENDPOINT = r.stdout.strip()
if not OPENAI_ENDPOINT.endswith('/'): OPENAI_ENDPOINT += '/'

r = subprocess.run(['az', 'cognitiveservices', 'account', 'keys', 'list', '-n', OPENAI_NAME, '-g', RESOURCE_GROUP,
    '--query', 'key1', '-o', 'tsv'], capture_output=True, text=True)
OPENAI_KEY = r.stdout.strip()

SEARCH_ENDPOINT = f'https://{SEARCH_NAME}.search.windows.net'
r = subprocess.run(['az', 'search', 'admin-key', 'show', '--service-name', SEARCH_NAME, '-g', RESOURCE_GROUP,
    '--query', 'primaryKey', '-o', 'tsv'], capture_output=True, text=True)
SEARCH_KEY = r.stdout.strip()

ctx = ssl.create_default_context()

print(f'\n✅ Resources detected:')
print(f'  OpenAI:  {OPENAI_NAME}')
print(f'  Search:  {SEARCH_NAME}')
print(f'  Storage: {STORAGE_NAME}')
print(f'  Endpoint: {OPENAI_ENDPOINT}')

## Azure CLI Authentication

Azure ML compute instances have a **managed identity** — this logs in instantly with no browser needed.

> If managed identity fails (permissions not assigned), the cell falls back to device code login.

In [ ]:
import subprocess, json as _j

# Check if already logged in
r = subprocess.run(['az', 'account', 'show'], capture_output=True, text=True)
if r.returncode == 0:
    acct = _j.loads(r.stdout)
    print(f'✅ Already logged in: {acct.get("user", {}).get("name", "unknown")}')
    print(f'   Subscription: {acct.get("name", "unknown")}')
else:
    # Try managed identity first (instant on Azure ML compute)
    r2 = subprocess.run(['az', 'login', '--identity'], capture_output=True, text=True)
    if r2.returncode == 0:
        acct = _j.loads(r2.stdout)[0]
        print(f'✅ Logged in via managed identity')
        print(f'   Subscription: {acct.get("name", "unknown")}')
    else:
        print('Managed identity not available. Using device code login...')
        !az login --use-device-code

## Step 2: Compare Search Modes

We'll run the same query through keyword, vector, and hybrid search to see how results differ.

### ✅ Prerequisite Check

This cell verifies that Lab 1 deployments (GPT-4o, embeddings) and Lab 2 search index exist before proceeding.

In [ ]:
# Verify all required resources exist before running search comparisons
import urllib.request, json as _check_json

print('Checking prerequisites...')
errors = []

# Check embedding deployment
try:
    url = f"{OPENAI_ENDPOINT}openai/deployments/text-embedding-3-small/embeddings?api-version=2024-06-01"
    data = _check_json.dumps({'input': 'test'}).encode()
    req = urllib.request.Request(url, data=data, headers={'Content-Type':'application/json','api-key':OPENAI_KEY})
    with urllib.request.urlopen(req, context=ctx) as resp:
        _check_json.loads(resp.read())
    print('  ✅ Embedding deployment (text-embedding-3-small) exists')
except Exception as e:
    errors.append('text-embedding-3-small')
    print(f'  ❌ Embedding deployment NOT FOUND: {e}')

# Check GPT-4o deployment
try:
    url = f"{OPENAI_ENDPOINT}openai/deployments/gpt-4o/chat/completions?api-version=2024-06-01"
    data = _check_json.dumps({'messages':[{'role':'user','content':'hi'}],'max_tokens':5}).encode()
    req = urllib.request.Request(url, data=data, headers={'Content-Type':'application/json','api-key':OPENAI_KEY})
    with urllib.request.urlopen(req, context=ctx) as resp:
        _check_json.loads(resp.read())
    print('  ✅ GPT-4o deployment exists')
except Exception as e:
    errors.append('gpt-4o')
    print(f'  ❌ GPT-4o deployment NOT FOUND: {e}')

# Check search index
try:
    url = f"{SEARCH_ENDPOINT}/indexes/banking-policies?api-version=2024-07-01"
    req = urllib.request.Request(url, headers={'api-key':SEARCH_KEY})
    with urllib.request.urlopen(req, context=ctx) as resp:
        _check_json.loads(resp.read())
    print('  ✅ Search index (banking-policies) exists')
except Exception as e:
    errors.append('banking-policies index')
    print(f'  ❌ Search index NOT FOUND: {e}')

if errors:
    print(f'\n⚠️  Missing: {", ".join(errors)}')
    print('   → Go back to Lab 1 and re-run Step 4 (GPT-4o) + Step 5 (embeddings)')
    print('   → Then re-run Lab 2 to create the search index')
    raise RuntimeError(f'Prerequisites missing: {", ".join(errors)}')
else:
    print('\n✅ All prerequisites verified. Ready for Lab 4.')

In [ ]:
def get_embedding(text):
    url = f"{OPENAI_ENDPOINT}openai/deployments/text-embedding-3-small/embeddings?api-version=2024-06-01"
    data = json.dumps({'input': text}).encode()
    req = urllib.request.Request(url, data=data, headers={'Content-Type':'application/json','api-key':OPENAI_KEY})
    with urllib.request.urlopen(req, context=ctx) as resp:
        return json.loads(resp.read())['data'][0]['embedding']

def search(query, mode='keyword', top_k=3, category_filter=None):
    url = f"{SEARCH_ENDPOINT}/indexes/banking-policies/docs/search?api-version=2024-07-01"
    body = {'select':'title,content,category,source_file','top':top_k}
    if category_filter: body['filter'] = f"category eq '{category_filter}'"
    if mode == 'keyword':
        body['search'] = query
    elif mode == 'vector':
        body['vectorQueries'] = [{'vector':get_embedding(query),'k':top_k,'fields':'content_vector','kind':'vector'}]
    elif mode == 'hybrid':
        body['search'] = query
        body['vectorQueries'] = [{'vector':get_embedding(query),'k':top_k,'fields':'content_vector','kind':'vector'}]
    data = json.dumps(body).encode()
    req = urllib.request.Request(url, data=data, method='POST',
        headers={'Content-Type':'application/json','api-key':SEARCH_KEY})
    with urllib.request.urlopen(req, context=ctx) as resp:
        return json.loads(resp.read()).get('value', [])

test_query = 'How can I reverse a payment I made?'
print('=' * 60)
print(f'  SEARCH MODE COMPARISON')
print(f'  Query: "{test_query}"')
print('=' * 60)

for mode in ['keyword', 'vector', 'hybrid']:
    results = search(test_query, mode=mode)
    print(f'\n📋 {mode.upper()} Search — {len(results)} results:')
    for r in results[:2]:
        print(f'   📄 {r["title"]} [{r["category"]}]')
        print(f'      {r["content"][:100]}...')

## Step 3: Filtered Search by Category

Azure AI Search supports filtering results by metadata fields like `category`.

In [ ]:
print('=' * 60)
print('  FILTERED SEARCH BY CATEGORY')
print('=' * 60)

for category in ['policy', 'product', 'security']:
    results = search('account information', mode='hybrid', category_filter=category)
    print(f'\n📁 Category: {category} — {len(results)} results')
    for r in results[:2]:
        print(f'   📄 {r["title"]}')

print('\n✅ Filtered search working.')

## Step 4: Build Conversational Search Engine

This engine maintains chat history across turns, rewrites follow-up questions, and generates grounded answers.

In [ ]:
def call_openai(messages, max_tokens=150, temperature=0.1):
    url = f"{OPENAI_ENDPOINT}openai/deployments/gpt-4o/chat/completions?api-version=2024-06-01"
    data = json.dumps({'messages':messages,'max_tokens':max_tokens,'temperature':temperature}).encode()
    req = urllib.request.Request(url, data=data, headers={'Content-Type':'application/json','api-key':OPENAI_KEY})
    with urllib.request.urlopen(req, context=ctx) as resp:
        return json.loads(resp.read())['choices'][0]['message']['content']

def hybrid_search(query, top_k=3):
    vector = get_embedding(query)
    url = f"{SEARCH_ENDPOINT}/indexes/banking-policies/docs/search?api-version=2024-07-01"
    body = {'search':query,
        'vectorQueries':[{'vector':vector,'k':top_k,'fields':'content_vector','kind':'vector'}],
        'select':'title,content,category,source_file','top':top_k}
    data = json.dumps(body).encode()
    req = urllib.request.Request(url, data=data, method='POST',
        headers={'Content-Type':'application/json','api-key':SEARCH_KEY})
    with urllib.request.urlopen(req, context=ctx) as resp:
        return json.loads(resp.read()).get('value', [])

class ConversationalSearch:
    def __init__(self):
        self.chat_history = []
        self.system_prompt = """You are a SecureBank conversational search assistant.
- Answer using ONLY the provided search results
- Cite sources using [Source: document_name] format
- If results don't contain the answer, say so clearly
- Suggest related follow-up questions"""
    
    def rewrite_for_search(self, question):
        if not self.chat_history: return question
        history = '\n'.join([f"Q: {h['q']}\nA: {h['a'][:100]}..." for h in self.chat_history[-3:]])
        return call_openai([
            {'role':'system','content':'Output only the rewritten search query.'},
            {'role':'user','content':f'Rewrite as standalone search query.\n\nConversation:\n{history}\n\nFollow-up: {question}\n\nStandalone:'}],
            max_tokens=50, temperature=0)
    
    def ask(self, question):
        print(f'\n{"─"*60}')
        print(f'👤 User: {question}')
        search_query = self.rewrite_for_search(question)
        if search_query != question: print(f'🔄 Search query: {search_query}')
        results = hybrid_search(search_query)
        print(f'🔍 Found {len(results)} documents')
        context = '\n\n'.join([f"[Source: {r['source_file']}] {r['title']}\n{r['content']}" for r in results])
        messages = [{'role':'system','content':self.system_prompt}]
        for h in self.chat_history[-3:]:
            messages.append({'role':'user','content':h['q']})
            messages.append({'role':'assistant','content':h['a']})
        messages.append({'role':'user','content':f'Search Results:\n{context}\n\nQuestion: {question}'})
        response = call_openai(messages)
        print(f'\n🤖 Assistant: {response}')
        self.chat_history.append({'q':question,'a':response})
        return response

print('✅ ConversationalSearch engine defined.')

## Step 5: Test — 6-Turn Banking Conversation

Watch how the engine maintains context across turns and rewrites follow-up questions.

In [ ]:
print('🏦 ' * 20)
print('  CONVERSATIONAL SEARCH — BANKING DEMO')
print('🏦 ' * 20)

engine = ConversationalSearch()

conversations = [
    'What savings accounts do you offer?',
    'Which one has the highest interest rate?',
    'Is there an early withdrawal penalty for CDs?',
    'What about wire transfers — what are the limits?',
    'Are international transfers more expensive?',
    'How do I report fraud on my account?',
]

for question in conversations:
    engine.ask(question)

print(f'\n{"─"*60}')
print(f'\n✅ Conversational search demo complete — {len(conversations)} turns.')

## Step 6: Search Analytics

In [ ]:
url = f"{SEARCH_ENDPOINT}/indexes/banking-policies/stats?api-version=2024-07-01"
req = urllib.request.Request(url, headers={'api-key':SEARCH_KEY})
with urllib.request.urlopen(req, context=ctx) as resp:
    stats = json.loads(resp.read())

print('📊 Index Statistics:')
print(f'   Documents: {stats.get("documentCount", "N/A")}')
print(f'   Storage:   {stats.get("storageSize", 0) / 1024:.1f} KB')

## ✅ Lab 4 Complete

**What you accomplished:**
- Compared keyword, vector, and hybrid search modes
- Implemented filtered search by category
- Built a conversational search engine with query rewriting
- Tested 6-turn multi-turn banking conversation

**Next:** Open `promptflow_lab5_rag_evaluation.ipynb`

> 🧹 **Cleanup:** Run the cleanup cell in **Lab 9** when all labs are complete to delete all resources.

---

© 2026 Great Learning. All rights reserved.